# Sentinel · fine-tune the reranker (hand-written PyTorch loop)

Trains `cross-encoder/ms-marco-MiniLM-L-6-v2` on the RFC training set in `data/reranker/` with `scripts/train_reranker.py` — no Trainer, no `.fit()`; the loop is in `sentinel/finetune/train.py`.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (roughly an hour; keep the tab open).
2. Three learning rates are tried; the winner is picked by **validation MRR@10 on the synthetic held-out RFCs** — never on the ground truth.
3. The winning rate is re-run with **every ground-truth passage excluded** from training (the strict no-leakage variant).
4. The last cell downloads `sentinel-reranker-runs.zip`: logs for every run and the two best checkpoints. Unzip it into the repo root (it creates `runs/`).

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
!test -d Sentinel || git clone --depth 1 --branch feat/reranker-finetune https://github.com/krishivsaini/Sentinel.git
%cd Sentinel
!git log --oneline -1

In [ ]:
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

**Check the loop first** — a tiny run (seconds) before spending GPU time.

In [ ]:
!python scripts/train_reranker.py --smoke --device cuda --out-dir runs/smoke

**Learning-rate sweep** — same seed and data; each run keeps its best checkpoint by validation MRR@10.

In [ ]:
LRS = ['1e-5', '2e-5', '4e-5']
for lr in LRS:
    !python scripts/train_reranker.py --lr {lr} --epochs 3 --eval-every 400 --out-dir runs/lr{lr}

In [ ]:
import json
results = {lr: json.load(open(f'runs/lr{lr}/summary.json')) for lr in LRS}
for lr, s in results.items():
    print(lr, 'best val MRR@10 %.4f at step %d of %d (%.0fs)' % (s['best_val']['mrr@10'], s['best_step'], s['steps'], s['seconds']))
BEST_LR = max(results, key=lambda lr: results[lr]['best_val']['mrr@10'])
print('chosen by validation:', BEST_LR)

**Strict variant** — the chosen rate, with every ground-truth passage removed from training (as positive and as negative).

In [ ]:
!python scripts/train_reranker.py --lr {BEST_LR} --epochs 3 --eval-every 400 --exclude-gold --out-dir runs/nogold

In [ ]:
import shutil, os
keep = ['runs/smoke/log.jsonl'] + [f'runs/lr{lr}/{f}' for lr in LRS for f in ('log.jsonl', 'summary.json')]
keep += ['runs/nogold/log.jsonl', 'runs/nogold/summary.json']
os.makedirs('export/runs', exist_ok=True)
for p in keep:
    os.makedirs(os.path.dirname('export/' + p), exist_ok=True); shutil.copy(p, 'export/' + p)
shutil.copytree(f'runs/lr{BEST_LR}/best', 'export/runs/main/best')
shutil.copy(f'runs/lr{BEST_LR}/log.jsonl', 'export/runs/main/log.jsonl')
shutil.copy(f'runs/lr{BEST_LR}/summary.json', 'export/runs/main/summary.json')
shutil.copytree('runs/nogold/best', 'export/runs/nogold/best')
json.dump({'chosen_lr': BEST_LR, 'by': 'validation MRR@10'}, open('export/runs/selection.json', 'w'))
shutil.make_archive('sentinel-reranker-runs', 'zip', 'export')
print(os.path.getsize('sentinel-reranker-runs.zip') // 2**20, 'MB')

In [ ]:
from google.colab import files
files.download('sentinel-reranker-runs.zip')